In [1]:
import numpy as np

In [2]:
data = [
    [12.0, 1.5, 1, 'Wine'],
    [5.0, 2.0, 0, 'Beer'],
    [40.0, 0.0, 1, 'Whiskey'],
    [13.5, 1.2, 1, 'Wine'],
    [4.5, 1.8, 0, 'Beer'],
    [38.0, 0.1, 1, 'Whiskey'],
    [11.5, 1.7, 1, 'Wine'],
    [5.5, 2.3, 0, 'Beer']
]


In [3]:
#Converting labels to integers
for i in data:
    if i[3]=='Wine':
        i.append(0)
    elif i[3]=='Beer':
        i.append(1)
    else:
        i.append(2)

In [4]:
#Converting the table into X (features) and y (labels) numpy arrays.
data = np.array(data)
X = np.array(data[:, :3], dtype= float)
y = np.array(data[:, 4], dtype= int)   


In [5]:
#Function to find gini value for a list of labels
def Gini(labels):
    values= np.unique(labels)
    counts=[]
    for i in values:
        counts.append(labels.count(i))
    sum=0.0
    for i in counts:
        sum= sum+i
    for i in range(len(counts)):
        counts[i]= counts[i]/float(sum)
    gini= 1
    for i in counts:
        gini= gini- np.square(i)
    return gini
Gini(['Wine', 'Beer', 'Whisky', 'Wine'])

np.float64(0.625)

In [14]:
#Function to find best split through gini values  
def best_split(X, y):
    feature = 0   #Gives index of the feature which gives best fit
    threshold = 0  #Gives threshold value
    gini_min = 1
    num_features = len(X[0])
    for counter in range(num_features):   # loop over features
        for row in X:
            j = row[counter]         # thresholds from that feature
            labelsL = []
            labelsR = []
            for idx in range(len(X)):
                if X[idx][counter] <= j:
                    labelsL.append(y[idx])
                else:
                    labelsR.append(y[idx])
            gini_Total = ((len(labelsL) * Gini(labelsL) / len(X)) +(len(labelsR) * Gini(labelsR) / len(X)))
            if gini_Total < gini_min:
                gini_min = gini_Total
                feature = counter
                threshold = j
    return feature, threshold   

In [17]:
class Node:
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, value=None):
        self.feature_index = feature_index  # which feature to split on
        self.threshold = threshold          
        self.left = left                    
        self.right = right                  
        self.value = value                  # only for leaf
def majority_class(y):
    return max(set(y), key=y.count)  
def build_tree(X, y, depth=0, max_depth=10, min_samples=1):
    # stop conditions
    if len(set(y)) == 1:
        return Node(value=y[0])
    if depth >= max_depth or len(y) < min_samples:
        return Node(value=majority_class(y))
    # find best split
    feature, threshold = best_split(X, y)
    X_left, X_right = [], []
    y_left, y_right = [], []
    for i in range(len(X)):
        if X[i][feature] <= threshold:
            X_left.append(X[i])
            y_left.append(y[i])
        else:
            X_right.append(X[i])
            y_right.append(y[i])
    
    # recursive calls
    left_child = build_tree(X_left, y_left, depth+1, max_depth, min_samples)
    right_child = build_tree(X_right, y_right, depth+1, max_depth, min_samples)
    
    return Node(feature, threshold, left_child, right_child)
    
def predict_one(node, x):
    if node.value== 0:
        return 'Wine'
    elif node.value==1:
        return 'Beer'
    elif node.value==2:
        return 'Whiskey'
    if x[node.feature_index] <= node.threshold:
        return predict_one(node.left, x)
    else:
        return predict_one(node.right, x)

def predict(tree, X):
    results = []
    for x in X:
        results.append(predict_one(tree, x))
    return results        

In [18]:
test_data = np.array([
    [6.0, 2.1, 0],   # Expected: Beer
    [39.0, 0.05, 1], # Expected: Whiskey
    [13.0, 1.3, 1]   # Expected: Wine
])
tree = build_tree(X, y)

predictions = predict(tree, test_data)

print(predictions)


['Wine', 'Whiskey', 'Wine']
